In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error,r2_score
import joblib

In [38]:
df=pd.read_csv("medical_insurance.csv")

In [39]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [40]:
df['region'].unique()

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [6]:
df.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(1435)

In [8]:
df.shape

(2772, 7)

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.shape

(1337, 7)

In [11]:
le=LabelEncoder()

In [12]:
df['region']=le.fit_transform(df['region'])
df['smoker']=le.fit_transform(df['smoker'])
df['sex']=le.fit_transform(df['sex'])

In [13]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


In [14]:
X=df.drop('charges', axis=1)
y=df['charges']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
scaler=StandardScaler()

In [17]:
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

In [18]:
# KNN_model=KNeighborsRegressor()

In [19]:
# KNN_model.fit(X_train_scaled, y_train).score(X_test_scaled, y_test)

In [20]:
# y_pre=KNN_model.predict(X_test_scaled)

In [21]:
# est=[
#     ('linear regresion',LinearRegression()),
#     # ('decisiontree regression', DecisionTreeRegressor()),
#     ('KNN', KNeighborsRegressor())
# ]

In [22]:
# stack_model=StackingRegressor(est, SVR(), cv=5)

In [23]:
# stack_model.fit(X_train_scaled, y_train).score(X_test_scaled, y_test)

In [24]:
estimators=[
        ('linear regresion',LinearRegression()),
        ('decisiontree regression', DecisionTreeRegressor()),
        ('KNN', KNeighborsRegressor())
    ] 

In [25]:
Models={
    'svr':SVR(),
    'linear_regression':LinearRegression(),
    'decisiontree_regression':DecisionTreeRegressor(max_depth=5),
    'stacking_regression':StackingRegressor(estimators=estimators, final_estimator=SVR(), cv=5),
    'randomforest_regression':RandomForestRegressor(n_estimators=100, random_state=42, max_depth=2),
    'adaboost_regression':AdaBoostRegressor(n_estimators=100, learning_rate=0.2, random_state=42),
    'gradientboost_regression':GradientBoostingRegressor(n_estimators=100, loss='quantile', learning_rate=0.1),
    'xgb_regression':XGBRegressor(n_estimators=100, learning_rate=0.1)
}

In [26]:
result=[]

In [27]:
for name,model in Models.items():
    model.fit(X_train_scaled, y_train)
    pre=model.predict(X_test_scaled)
    MAE=mean_absolute_error(y_test, pre)
    R2_score=r2_score(y_test, pre)
    result.append({
        name:{'MAE':MAE, 'R2_score':R2_score}
    })

In [28]:
print()

In [29]:
result

[{'svr': {'MAE': 9252.823195715733, 'R2_score': -0.12997270612737677}},
 {'linear_regression': {'MAE': 4134.64345710585,
   'R2_score': 0.7917732244147461}},
 {'decisiontree_regression': {'MAE': 2583.555340214416,
   'R2_score': 0.892484639486742}},
 {'stacking_regression': {'MAE': 9180.97391902793,
   'R2_score': -0.11677718733536735}},
 {'randomforest_regression': {'MAE': 3197.438802497311,
   'R2_score': 0.868625308163498}},
 {'adaboost_regression': {'MAE': 4233.480608788573,
   'R2_score': 0.8619344475121298}},
 {'gradientboost_regression': {'MAE': 6296.090174574624,
   'R2_score': 0.7092634558090267}},
 {'xgb_regression': {'MAE': 8056.9984352613255,
   'R2_score': -0.15968003445361356}}]

In [30]:
joblib.dump(Models['decisiontree_regression'], 'decision_tree_regresion.pkl')
joblib.dump(scaler, 'scal.pkl')
joblib.dump(X.columns.tolist(), 'column.pkl')

['column.pkl']

In [31]:
X.columns.tolist()

['age', 'sex', 'bmi', 'children', 'smoker', 'region']

In [36]:
df['region'].unique()

array([3, 2, 1, 0])

In [34]:
df['bmi'].max()

53.13